In [ ]:
import pandas as pd
import re
import spacy
from IPython.display import YouTubeVideo

In [ ]:
with open(file = 'Avatar_S3_E21_Transcript.txt', mode = 'r', encoding = 'utf-8') as file:
    transcript = file.read() 

In [ ]:
rows = []

# For each line in the transcript, strip() removes extra whitespace/newlines at the start & end. 
# split("\n") separates the text into individual lines
for line in transcript.strip().split("\n"):

    # Split the line into two parts using 2 or more spaces as the separator. r"\s{2,}" means any whitespace character occurring 2 or more times.
    # maxsplit=1 ensures the line is only split once
    parts = re.split(r"\s{2,}", line, maxsplit=1)
    
    if len(parts) == 2:
        character = parts[0]

        # Remove bracketed text (non-dialogue) from dialogue
        dialogue = re.sub(r"\[.*?\]", "", parts[1])

        # Clean up extra whitespace
        dialogue = re.sub(r"\s+", " ", dialogue).strip()

        rows.append([character, dialogue])

transcript_df = pd.DataFrame(rows, columns=["Character", "Dialogue"])

In [ ]:
# Save the dataframe
transcript_df.to_pickle('transcript_df.pkl')

In [ ]:
transcript_df = pd.read_pickle('transcript_df.pkl')

In [ ]:
# Plot a bar graph
transcript_df["Character"].value_counts().plot(kind = "bar", title = "Frequency Of Character Dialogue", ylabel = "Frequency")

Funnily enough, Aang isn't even in the top three even though it's the series finale 😆

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Create a new column called Dialogue Doc, which is then passed through the spacy pipeline for analysis.
transcript_df["Dialogue Doc"] = transcript_df["Dialogue"].apply(nlp)

# Add token count column
transcript_df["Token Count"] = transcript_df["Dialogue Doc"].apply(len)

In [ ]:
transcript_df

In [ ]:
# Get the top three speakers -> Get the labels (which are the index of the series) -> Convert to a list
top_speakers = (
    transcript_df["Character"]
    .value_counts()
    .head(3)
    .index
    .tolist()
)

In [ ]:
# Filter by the top three speakers
top_speaker_df = transcript_df[transcript_df["Character"].isin(top_speakers)]

In [ ]:
# Pick the longest dialogue for each top speaker
top_speaker_df = top_speaker_df.sort_values("Token Count", ascending=False).drop_duplicates("Character")

In [ ]:
top_speaker_df

In [ ]:
# Tokens & morphology of each top speaker's longest dialogue.

# iterrows() return the row index & the row data.
# The loop receives these two values each time, but the underscore means that the first value (row index) is ignored.
for _, row in top_speaker_df.iterrows():

    print(f"\nCharacter: {row['Character']}")

    for token in row["Dialogue Doc"]:
        
        # Skip punctuation mark tokens
        if token.is_punct == True:
            pass
        else:
            
            # Print tokens + morphology
            print(
                f"Token: {token.text:<15} "
                f"Morphology: {token.morph}"
            )

In [ ]:
# Named Entity Recognition of each top speaker's longest dialogue
for _, row in top_speaker_df.iterrows():
    
    print(f"\n{row['Character']} mentioned these named entities:")

    if not row["Dialogue Doc"].ents:
            print("None")
    else:
        for ent in row["Dialogue Doc"].ents:
    
        # Print named entites & their labels
            print(
                f"{ent.text:<15} "
                f"Label/Type: {ent.label_}"
            )

It might seem weird that Sokka has not mentioned any named entities, but if we run the cell below, we can see that is indeed the case.

In [ ]:
for _, row in top_speaker_df.iterrows():
    print(f"{row['Character']}")
    print(f"{row['Dialogue']}\n")

In [ ]:
# Enjoy this video of the season finale 🔥
YouTubeVideo('kXShLPXfWZA', height = 480, width = 854, start = 390)

In [ ]:
# Print dependencies
%load_ext watermark

%watermark -v -m -p pandas,re,spacy,watermark

# Date
print(" ")
%watermark -u -n -t -z